# Data Inspection

Purpose: this notebook validates the raw dataset, verifies data integrity, and documents the transformation pipeline before exploratory analysis.

In [1]:
import pandas as pd

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_processing import (
    load_raw_data,
    validate_columns,
    validate_data_types,
    validate_missing_values,
    validate_duplicates,
    validate_participants,
    validate_trials,
    validate_variables,
    apply_transformations,
)

In [2]:
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "vandenberg12.csv"

In [3]:
# Load raw dataset

df = load_raw_data(RAW_PATH)

print(df.head())

  experiment  id  trial   stims response_selection  setsize  target  response  \
0      ExpS3   1      1  colors         colorwheel        2      72        76   
1      ExpS3   1      2  colors         colorwheel        6      46       104   
2      ExpS3   1      3  colors         colorwheel        1     164       118   
3      ExpS3   1      4  colors         colorwheel        7     358       348   
4      ExpS3   1      5  colors         colorwheel        6     292       316   

   targetrad  responserad    devrad  errorrad  
0   1.256637     1.326450  0.069813  0.069813  
1   0.802851     1.815142  1.012291  1.012291  
2   2.862340     2.059489 -0.802851  0.802851  
3   6.248279     6.073746 -0.174533  0.174533  
4   5.096361     5.515240  0.418879  0.418879  


In [4]:
# Dataset overview

print("Dataset shape:", df.shape)

df.info()

print(df.describe(include="all"))

Dataset shape: (37824, 12)
<class 'pandas.DataFrame'>
RangeIndex: 37824 entries, 0 to 37823
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   experiment          37824 non-null  str    
 1   id                  37824 non-null  int64  
 2   trial               37824 non-null  int64  
 3   stims               37824 non-null  str    
 4   response_selection  37824 non-null  str    
 5   setsize             37824 non-null  int64  
 6   target              37824 non-null  int64  
 7   response            37824 non-null  int64  
 8   targetrad           37824 non-null  float64
 9   responserad         37824 non-null  float64
 10  devrad              37824 non-null  float64
 11  errorrad            37824 non-null  float64
dtypes: float64(4), int64(5), str(3)
memory usage: 3.5 MB
       experiment            id         trial   stims response_selection  \
count       37824  37824.000000  37824.000000   37824  

In [5]:
# Validations

validation_results = {
    "columns": validate_columns(df),
    "data_types": validate_data_types(df),
    "missing_values": validate_missing_values(df),
    "duplicates": validate_duplicates(df),
    "participants": validate_participants(df),
    "trials": validate_trials(df),
    "variables": validate_variables(df),
}

print("Dataset validation completed successfully.")
print(validation_results["participants"])
print(validation_results["trials"])

Dataset validation completed successfully.
{'total_participants': 13, 'participants_by_experiment': {'Exp1': 13, 'Exp2': 6, 'ExpS3': 13}}
{'total_trials': 37824, 'trials_per_participant_min': 864, 'trials_per_participant_max': 2560, 'mean_trials_per_participant': 1182.0}


In [6]:
# Experiment comparison

print("Response selection by experiment:")
print(
    pd.crosstab(
        df["experiment"],
        df["response_selection"]
    )
)

print("Error distribution by experiment:")
print(
    df.groupby("experiment")["errorrad"]
    .describe()
)

Response selection by experiment:
response_selection  colorwheel  rotation  scrolling
experiment                                         
Exp1                         0         0      11232
Exp2                         0     15360          0
ExpS3                    11232         0          0
Error distribution by experiment:
              count      mean       std  min       25%       50%       75%  \
experiment                                                                   
Exp1        11232.0  0.564463  0.651829  0.0  0.139626  0.349066  0.698132   
Exp2        15360.0  0.679142  0.762828  0.0  0.157080  0.383972  0.837758   
ExpS3       11232.0  0.648410  0.754089  0.0  0.139626  0.349066  0.837758   

                 max  
experiment            
Exp1        3.141593  
Exp2        3.141593  
ExpS3       3.141593  


In [7]:
# Apply transformations

processed_df = apply_transformations(df)

print(processed_df.head())

  experiment  id  trial   stims response_selection  setsize  target  response  \
0       Exp1   1      1  colors          scrolling        3     300       296   
1       Exp1   1      2  colors          scrolling        4     278       274   
2       Exp1   1      3  colors          scrolling        2     168       150   
3       Exp1   1      4  colors          scrolling        1     326       334   
4       Exp1   1      5  colors          scrolling        3     294       302   

   targetrad  responserad    devrad  errorrad  
0   5.235988     5.166175 -0.069813  0.069813  
1   4.852015     4.782202 -0.069813  0.069813  
2   2.932153     2.617994 -0.314159  0.314159  
3   5.689773     5.829400  0.139626  0.139626  
4   5.131268     5.270894  0.139626  0.139626  


In [8]:
# Processed dataset inspection

print(processed_df.describe())

processed_df.info()

                 id         trial       setsize        target      response  \
count  37824.000000  37824.000000  37824.000000  37824.000000  37824.000000   
mean       5.578680    776.865482      4.500000    184.375793    187.254706   
std        3.529016    657.397587      2.291318    103.503902    103.346462   
min        1.000000      1.000000      1.000000      1.000000      1.000000   
25%        3.000000    296.000000      2.750000     97.000000     96.000000   
50%        5.000000    591.500000      4.500000    186.000000    190.000000   
75%        8.000000    984.250000      6.250000    274.000000    273.000000   
max       13.000000   2560.000000      8.000000    360.000000    360.000000   

          targetrad   responserad        devrad      errorrad  
count  37824.000000  37824.000000  37824.000000  37824.000000  
mean       3.217965      3.268211     -0.029988      0.635962  
std        1.806484      1.803736      0.968087      0.730503  
min        0.017453      0.01745

In [9]:
# Save processed dataset

processed_path = PROJECT_ROOT / "data" / "processed" / "vandenberg12-clean.csv"

processed_path.parent.mkdir(parents=True, exist_ok=True)

processed_df.to_csv(
    processed_path,
    index=False
)

print("Saved:", processed_path)

Saved: F:\doc-hub\education\psychology\projects\research\working-memory-bayesian-model\data\processed\vandenberg12-clean.csv
